In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain.agents import AgentState

class customstate(AgentState):
    favourite_colour: str

<h2>write to state</h2>

In [5]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they mention about it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

model= ChatGoogleGenerativeAI(model="gemini-3.6-flash")

agent = create_agent(
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=customstate
)

In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

from pprint import pprint
pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='572f07b6-8838-430f-a0d2-fa799d33adc9'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_3711588569081115581': 'EswDCskDAWkUfRNRwBIO34+NiICCjYmUDOQh31M7yaZkkxNu8YrdS3bMCq+TffDNoY1X9YTvBDFLIaKOFCQtxnwc9YjgxVnmXEN2y+3Lvkxy7tsozIBSvIqG4xHfWX4xJd87eVBtSsXlROMx/2zatDPawByyLVtlVxd2oFZNp0aUepipSF8s3Vz3ul/U5TUdUXVJf3CBCJqnHQbLSoJ0QN/AxojkZY60OXYNVC0lLkByu7g+8JuzC9CiLj15XqutDC7hk3iADRT5PjT4mvFh9mE9/dMXEgS9pQGd0U4E/0W0ZJenMasL/U22LkF67FHQL5GImgQFGBgVpwbEJ2ZQBJqJQjzEnnCv9RBVVMt2gdW5XxNyCWLBmvl10SaVRRLQXrwBIoOK/zW3lWVn6eK4eexysc+DRCpMCBK39tGWc+MBXRj9tij6eAhhfuyfdUDc1ZAmGGPZZSa/Wam9tRwCmfWI8dbHIuGJ+f4zSC0dF0Y7FSaXyKdiDPVQOwLlXFjEceo87Gl64MY19QQBgSwCgmgQ996GunuG/v5QtX7XGzqP7Cjvh3flmz7B5wjGcJjdF+fnfSn

In [ ]:
# we can also update directly while invoking
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'blue',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='fca8a4d9-5ea7-4dc3-a400-3966a36c505a'),
              AIMessage(content=[{'type': 'text', 'text': "Hello! I'm doing well, thank you for asking. How are you doing today? How can I help you?", 'extras': {'signature': 'EvICCu8CAWkUfRNDgt9dhi0ZRwM6w0CdejNPFQln8Ubrm3UBSR3i5nleTLPko8x8Bs9I5epeQG0ed+/K2kRaTYThP+0b9WSSwcNESIjvXlBJ8Ww9DQv1FN3YudAdmue9rLQ/O/NVcoEjD+xMH5Uz8p0kQxsVPhatV1SEXN5dpfbGFaHYCCrqaSfCI/LFNcm4EEmRETW1FqPZSbIStksvXDUHb2QqSxkaydfSeauvPiDdZMtFZoOps/vZX1evVn8cjEErbi1mPXdvvdWjlvtDrtwXRoQVVbeViske4MQ9hSA5XRCD4TuRLWFVLdjnHJK2h7PvSckYWkdHNPxOx16zSINTRKs8vO60MhBCpjip6uOItUp0lxxkJ3o2C/7a4T2Pqwb6ocEs4ee5nmQ8kp1Zjiz3lprwiEYa+fK0RABi5Rii8B8cXq3tdYLkiz0vyZZdLxoWedIl1IjxqXXR5N6Rxt/koRSVU4qyF8O8HXoIdd07nT1gSQ=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'go

In [12]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-20b",
    
)
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=customstate
)

In [13]:
response = agent.invoke(
    { "messages": [HumanMessage(content="my favourite colour is blue")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'blue',
 'messages': [HumanMessage(content='my favourite colour is blue', additional_kwargs={}, response_metadata={}, id='e01e0eac-b7ea-45be-9066-10f6c97c6159'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'User says favourite colour is blue. We need to call update_favourite_colour with favourite_colour: "blue".', 'tool_calls': [{'id': 'fc_a271e089-c8a3-4e7c-8347-abaea3a47041', 'function': {'arguments': '{"favourite_colour":"blue"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 162, 'total_tokens': 212, 'completion_time': 0.078745142, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.009890677, 'prompt_tokens_details': None, 'queue_time': 0.285013468, 'total_time': 0.088635819}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs':

In [14]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'blue',
 'messages': [HumanMessage(content='my favourite colour is blue', additional_kwargs={}, response_metadata={}, id='e01e0eac-b7ea-45be-9066-10f6c97c6159'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'User says favourite colour is blue. We need to call update_favourite_colour with favourite_colour: "blue".', 'tool_calls': [{'id': 'fc_a271e089-c8a3-4e7c-8347-abaea3a47041', 'function': {'arguments': '{"favourite_colour":"blue"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 162, 'total_tokens': 212, 'completion_time': 0.078745142, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.009890677, 'prompt_tokens_details': None, 'queue_time': 0.285013468, 'total_time': 0.088635819}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs':

In [15]:
response = agent.invoke(
    { "messages": [HumanMessage(content="my favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='my favourite colour is blue', additional_kwargs={}, response_metadata={}, id='e01e0eac-b7ea-45be-9066-10f6c97c6159'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'User says favourite colour is blue. We need to call update_favourite_colour with favourite_colour: "blue".', 'tool_calls': [{'id': 'fc_a271e089-c8a3-4e7c-8347-abaea3a47041', 'function': {'arguments': '{"favourite_colour":"blue"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 162, 'total_tokens': 212, 'completion_time': 0.078745142, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.009890677, 'prompt_tokens_details': None, 'queue_time': 0.285013468, 'total_time': 0.088635819}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs'

In [16]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='my favourite colour is blue', additional_kwargs={}, response_metadata={}, id='e01e0eac-b7ea-45be-9066-10f6c97c6159'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'User says favourite colour is blue. We need to call update_favourite_colour with favourite_colour: "blue".', 'tool_calls': [{'id': 'fc_a271e089-c8a3-4e7c-8347-abaea3a47041', 'function': {'arguments': '{"favourite_colour":"blue"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 162, 'total_tokens': 212, 'completion_time': 0.078745142, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.009890677, 'prompt_tokens_details': None, 'queue_time': 0.285013468, 'total_time': 0.088635819}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs'